In [0]:
# NOTES for updates: 
# vbfa does not contain POSNV
# resb does not contain RSART
# stpo does not have STZHL

In [0]:
catalog = "sample_synthetic_sap"
schema = "sap"

In [ ]:
# Cleanup Cell

tables = [
    "afko", "kna1", "likp", "lips", "makt", "mara", "marc",
    "mard", "marm", "mast", "matdoc", "mbew", "resb", "stpo", "vbak",
    "vbap", "vbep", "vbfa"
]

# 1. Drop Foreign Keys first (to avoid dependency errors)
print(f"--- Dropping Foreign Keys in {catalog}.{schema} ---")
for t in tables:
    # Query Information Schema for Foreign Keys on this table
    fk_df = spark.sql(f"""
        SELECT constraint_name 
        FROM {catalog}.information_schema.table_constraints 
        WHERE table_schema = '{schema}' 
          AND table_name = '{t}' 
          AND constraint_type = 'FOREIGN KEY'
    """).collect()
    
    for row in fk_df:
        try:
            print(f"Dropping FK {row['constraint_name']} from {t}")
            spark.sql(f"ALTER TABLE {catalog}.{schema}.{t} DROP CONSTRAINT {row['constraint_name']}")
        except Exception as e:
            print(f"Skipped {row['constraint_name']}: {e}")

# 2. Drop Primary Keys next
print(f"\n--- Dropping Primary Keys in {catalog}.{schema} ---")
for t in tables:
    # Query Information Schema for Primary Keys on this table
    pk_df = spark.sql(f"""
        SELECT constraint_name 
        FROM {catalog}.information_schema.table_constraints 
        WHERE table_schema = '{schema}' 
          AND table_name = '{t}' 
          AND constraint_type = 'PRIMARY KEY'
    """).collect()
    
    for row in pk_df:
        try:
            print(f"Dropping PK {row['constraint_name']} from {t}")
            # Note: Databricks allows dropping PK without name, but dropping by constraint name is safer if known
            spark.sql(f"ALTER TABLE {catalog}.{schema}.{t} DROP PRIMARY KEY")
        except Exception as e:
            print(f"Skipped PK on {t}: {e}")

print("\n--- State Cleared. Ready for Constraint Creation. ---")

In [ ]:
%python

# Dictionary of Table Name -> List of PK Columns that must be NOT NULL
pk_definitions = {
    "mara":   ["MANDT", "MATNR"],
    "marc":   ["MANDT", "MATNR", "WERKS"],
    "mard":   ["MANDT", "MATNR", "WERKS", "LGORT"],
    "marm":   ["MANDT", "MATNR", "MEINH"],
    "makt":   ["MANDT", "MATNR", "SPRAS"],
    "mbew":   ["MANDT", "MATNR", "BWKEY"],
    "kna1":   ["MANDT", "KUNNR"],
    "vbak":   ["MANDT", "VBELN"],
    "vbap":   ["MANDT", "VBELN", "POSNR"],
    "vbep":   ["MANDT", "VBELN", "POSNR", "ETENR"],
    "vbfa":   ["MANDT", "VBELV", "POSNV", "VBELN", "POSNN", "VBTYP_N"],
    "likp":   ["MANDT", "VBELN"],
    "lips":   ["MANDT", "VBELN", "POSNR"],
    "afko":   ["MANDT", "AUFNR"],
    "mast":   ["MANDT", "MATNR", "WERKS", "STLAN"],
    "stpo":   ["MANDT", "STLTY", "STLNR", "STLKN", "STZHL"],
    "resb":   ["MANDT", "RSNUM", "RSPOS", "RSART"],
    "matdoc": ["MANDT", "MBLNR", "MJAHR", "ZEILE"]
}

print(f"--- ENFORCING NOT NULL ON PK COLUMNS IN {catalog}.{schema} ---")

for table, columns in pk_definitions.items():
    for col in columns:
        try:
            # Construct the ALTER command
            # Syntax: ALTER TABLE table_name ALTER COLUMN col_name SET NOT NULL
            cmd = f"ALTER TABLE {catalog}.{schema}.{table} ALTER COLUMN {col} SET NOT NULL"
            spark.sql(cmd)
            print(f"Set NOT NULL: {table}.{col}")
            
        except Exception as e:
            # If the data actually contains NULLs, this will fail with a specific error.
            if "contains null values" in str(e).lower():
                print(f"CRITICAL ERROR: Could not set {table}.{col} to NOT NULL because it contains null data.")
            else:
                # If it's already not null or another issue, just print it
                print(f"Note on {table}.{col}: {e}")

print("--- NOT NULL ENFORCEMENT COMPLETE ---")

In [ ]:
%sql
-- ==========================================
-- 1. PRIMARY KEYS
-- ==========================================
ALTER TABLE sample_synthetic_sap.sap.mara ADD CONSTRAINT pk_mara PRIMARY KEY (MANDT, MATNR);
ALTER TABLE sample_synthetic_sap.sap.marc ADD CONSTRAINT pk_marc PRIMARY KEY (MANDT, MATNR, WERKS);
ALTER TABLE sample_synthetic_sap.sap.mard ADD CONSTRAINT pk_mard PRIMARY KEY (MANDT, MATNR, WERKS, LGORT);
ALTER TABLE sample_synthetic_sap.sap.marm ADD CONSTRAINT pk_marm PRIMARY KEY (MANDT, MATNR, MEINH);
ALTER TABLE sample_synthetic_sap.sap.makt ADD CONSTRAINT pk_makt PRIMARY KEY (MANDT, MATNR, SPRAS);
ALTER TABLE sample_synthetic_sap.sap.mbew ADD CONSTRAINT pk_mbew PRIMARY KEY (MANDT, MATNR, BWKEY);
ALTER TABLE sample_synthetic_sap.sap.kna1 ADD CONSTRAINT pk_kna1 PRIMARY KEY (MANDT, KUNNR);
ALTER TABLE sample_synthetic_sap.sap.vbak ADD CONSTRAINT pk_vbak PRIMARY KEY (MANDT, VBELN);
ALTER TABLE sample_synthetic_sap.sap.vbap ADD CONSTRAINT pk_vbap PRIMARY KEY (MANDT, VBELN, POSNR);
ALTER TABLE sample_synthetic_sap.sap.vbep ADD CONSTRAINT pk_vbep PRIMARY KEY (MANDT, VBELN, POSNR, ETENR);
--ALTER TABLE sample_synthetic_sap.sap.vbfa ADD CONSTRAINT pk_vbfa PRIMARY KEY (MANDT, VBELV, POSNV, VBELN, POSNN, VBTYP_N);
ALTER TABLE sample_synthetic_sap.sap.likp ADD CONSTRAINT pk_likp PRIMARY KEY (MANDT, VBELN);
ALTER TABLE sample_synthetic_sap.sap.lips ADD CONSTRAINT pk_lips PRIMARY KEY (MANDT, VBELN, POSNR);
ALTER TABLE sample_synthetic_sap.sap.afko ADD CONSTRAINT pk_afko PRIMARY KEY (MANDT, AUFNR);
ALTER TABLE sample_synthetic_sap.sap.mast ADD CONSTRAINT pk_mast PRIMARY KEY (MANDT, MATNR, WERKS, STLAN);
--ALTER TABLE sample_synthetic_sap.sap.stpo ADD CONSTRAINT pk_stpo PRIMARY KEY (MANDT, STLTY, STLNR, STLKN, STZHL);
--ALTER TABLE sample_synthetic_sap.sap.resb ADD CONSTRAINT pk_resb PRIMARY KEY (MANDT, RSNUM, RSPOS, RSART);
ALTER TABLE sample_synthetic_sap.sap.matdoc ADD CONSTRAINT pk_matdoc PRIMARY KEY (MANDT, MBLNR, MJAHR, ZEILE);

-- ==========================================
-- 2. FOREIGN KEYS
-- ==========================================

-- Material & Customer
ALTER TABLE sample_synthetic_sap.sap.marc ADD CONSTRAINT fk_marc_mara FOREIGN KEY (MANDT, MATNR) REFERENCES sample_synthetic_sap.sap.mara (MANDT, MATNR);
ALTER TABLE sample_synthetic_sap.sap.mard ADD CONSTRAINT fk_mard_marc FOREIGN KEY (MANDT, MATNR, WERKS) REFERENCES sample_synthetic_sap.sap.marc (MANDT, MATNR, WERKS);
ALTER TABLE sample_synthetic_sap.sap.marm ADD CONSTRAINT fk_marm_mara FOREIGN KEY (MANDT, MATNR) REFERENCES sample_synthetic_sap.sap.mara (MANDT, MATNR);
ALTER TABLE sample_synthetic_sap.sap.makt ADD CONSTRAINT fk_makt_mara FOREIGN KEY (MANDT, MATNR) REFERENCES sample_synthetic_sap.sap.mara (MANDT, MATNR);
ALTER TABLE sample_synthetic_sap.sap.mbew ADD CONSTRAINT fk_mbew_mara FOREIGN KEY (MANDT, MATNR) REFERENCES sample_synthetic_sap.sap.mara (MANDT, MATNR);

-- Sales
ALTER TABLE sample_synthetic_sap.sap.vbak ADD CONSTRAINT fk_vbak_kna1 FOREIGN KEY (MANDT, KUNNR) REFERENCES sample_synthetic_sap.sap.kna1 (MANDT, KUNNR);
ALTER TABLE sample_synthetic_sap.sap.vbap ADD CONSTRAINT fk_vbap_vbak FOREIGN KEY (MANDT, VBELN) REFERENCES sample_synthetic_sap.sap.vbak (MANDT, VBELN);
ALTER TABLE sample_synthetic_sap.sap.vbap ADD CONSTRAINT fk_vbap_mara FOREIGN KEY (MANDT, MATNR) REFERENCES sample_synthetic_sap.sap.mara (MANDT, MATNR);
ALTER TABLE sample_synthetic_sap.sap.vbep ADD CONSTRAINT fk_vbep_vbap FOREIGN KEY (MANDT, VBELN, POSNR) REFERENCES sample_synthetic_sap.sap.vbap (MANDT, VBELN, POSNR);

-- Logistics
ALTER TABLE sample_synthetic_sap.sap.likp ADD CONSTRAINT fk_likp_kna1 FOREIGN KEY (MANDT, KUNNR) REFERENCES sample_synthetic_sap.sap.kna1 (MANDT, KUNNR);
ALTER TABLE sample_synthetic_sap.sap.lips ADD CONSTRAINT fk_lips_likp FOREIGN KEY (MANDT, VBELN) REFERENCES sample_synthetic_sap.sap.likp (MANDT, VBELN);
ALTER TABLE sample_synthetic_sap.sap.lips ADD CONSTRAINT fk_lips_mara FOREIGN KEY (MANDT, MATNR) REFERENCES sample_synthetic_sap.sap.mara (MANDT, MATNR);

-- Production & BOM
ALTER TABLE sample_synthetic_sap.sap.mast ADD CONSTRAINT fk_mast_mara FOREIGN KEY (MANDT, MATNR) REFERENCES sample_synthetic_sap.sap.mara (MANDT, MATNR);
ALTER TABLE sample_synthetic_sap.sap.mast ADD CONSTRAINT fk_mast_marc FOREIGN KEY (MANDT, MATNR, WERKS) REFERENCES sample_synthetic_sap.sap.marc (MANDT, MATNR, WERKS);
ALTER TABLE sample_synthetic_sap.sap.stpo ADD CONSTRAINT fk_stpo_mara FOREIGN KEY (MANDT, IDNRK) REFERENCES sample_synthetic_sap.sap.mara (MANDT, MATNR);
ALTER TABLE sample_synthetic_sap.sap.resb ADD CONSTRAINT fk_resb_mara FOREIGN KEY (MANDT, MATNR) REFERENCES sample_synthetic_sap.sap.mara (MANDT, MATNR);
ALTER TABLE sample_synthetic_sap.sap.resb ADD CONSTRAINT fk_resb_afko FOREIGN KEY (MANDT, AUFNR) REFERENCES sample_synthetic_sap.sap.afko (MANDT, AUFNR);

-- MATDOC
-- Link to General Material Master
ALTER TABLE sample_synthetic_sap.sap.matdoc 
ADD CONSTRAINT fk_matdoc_mara FOREIGN KEY (MANDT, MATNR) 
REFERENCES sample_synthetic_sap.sap.mara (MANDT, MATNR);

-- Link to Plant Material Master (Validates the Material exists in this Plant)
-- This is stronger than just linking to MARA because it validates the WERKS too
ALTER TABLE sample_synthetic_sap.sap.matdoc 
ADD CONSTRAINT fk_matdoc_marc FOREIGN KEY (MANDT, MATNR, WERKS) 
REFERENCES sample_synthetic_sap.sap.marc (MANDT, MATNR, WERKS);

-- Link to Production Orders (If AUFNR is populated)
-- Note: This is standard for movement types 101 (GR for Order) or 261 (GI for Order)
ALTER TABLE sample_synthetic_sap.sap.matdoc 
ADD CONSTRAINT fk_matdoc_afko FOREIGN KEY (MANDT, AUFNR) 
REFERENCES sample_synthetic_sap.sap.afko (MANDT, AUFNR);

-- Link to Customer (If KUNNR is populated)
-- Note: Standard for movement types 601 (Goods Issue to Customer)
ALTER TABLE sample_synthetic_sap.sap.matdoc 
ADD CONSTRAINT fk_matdoc_kna1 FOREIGN KEY (MANDT, KUNNR) 
REFERENCES sample_synthetic_sap.sap.kna1 (MANDT, KUNNR);
/*
-- Link to Delivery (If VBELN_IM or VBELN is used - assuming VBELN based on your schema)
-- Note: Check if your matdoc table uses 'VBELN' or 'VBELN_IM' for the delivery number. 
-- I am using VBELN here to match your LIKP key.
ALTER TABLE sample_synthetic_sap.sap.matdoc 
ADD CONSTRAINT fk_matdoc_likp FOREIGN KEY (MANDT, VBELN) 
REFERENCES sample_synthetic_sap.sap.likp (MANDT, VBELN);*/

In [0]:
import pandas as pd

# CONFIGURATION

table_group_color = "#1f77b4"
output_path = f"dbfs:/Volumes/sap_mock_data/sap/files/SE16 - MAKT.CSV{catalog}_{schema}.dbml"

print(f"--- Generating DBML for {catalog}.{schema} ---")

# 1. GET COLUMNS
cols_query = f"""
    SELECT table_name, column_name, data_type, comment 
    FROM {catalog}.information_schema.columns 
    WHERE table_schema = '{schema}'
    ORDER BY table_name, ordinal_position
"""
cols_df = spark.sql(cols_query).toPandas()

# 2. GET FOREIGN KEYS (Fixed Query)
# We join key_column_usage to itself to map Source -> Target by Position
fk_query = f"""
    SELECT 
        src.table_name AS source_table,
        tgt.table_name AS target_table,
        rc.constraint_name,
        -- We collect structs of (position, name), sort by position, then extract name
        -- This ensures (MANDT, MATNR) stays in that order and doesn't flip to (MATNR, MANDT)
        concat('(', array_join(transform(array_sort(collect_list(struct(src.ordinal_position, src.column_name))), x -> x.column_name), ', '), ')') AS source_cols,
        concat('(', array_join(transform(array_sort(collect_list(struct(tgt.ordinal_position, tgt.column_name))), x -> x.column_name), ', '), ')') AS target_cols
    FROM {catalog}.information_schema.referential_constraints rc
    -- Join for Source Columns (The FK table)
    JOIN {catalog}.information_schema.key_column_usage src 
        ON rc.constraint_name = src.constraint_name 
        AND rc.constraint_schema = src.constraint_schema
    -- Join for Target Columns (The PK table)
    JOIN {catalog}.information_schema.key_column_usage tgt 
        ON rc.unique_constraint_name = tgt.constraint_name 
        AND rc.unique_constraint_schema = tgt.constraint_schema
        AND src.ordinal_position = tgt.ordinal_position -- Critical: Match 1st col to 1st col
    WHERE rc.constraint_schema = '{schema}'
    GROUP BY src.table_name, tgt.table_name, rc.constraint_name
"""
fks_df = spark.sql(fk_query).toPandas()

# 3. BUILD DBML
dbml_lines = []

# A. Header
dbml_lines.append(f"Project {catalog}_{schema} {{")
dbml_lines.append(f"  database_type: 'Databricks'")
dbml_lines.append(f"  Note: 'Auto-generated SAP ERD'")
dbml_lines.append("}}\n")

# B. Table Group
unique_tables = cols_df['table_name'].unique()
dbml_lines.append(f"TableGroup \"{schema}_tables\" {{")
for t in unique_tables:
    dbml_lines.append(f"  \"{t}\"")
dbml_lines.append("}\n")

# C. Tables
for table_name in unique_tables:
    table_cols = cols_df[cols_df['table_name'] == table_name]
    dbml_lines.append(f"Table \"{table_name}\" [headercolor: {table_group_color}] {{")
    for _, row in table_cols.iterrows():
        col_def = f"  \"{row['column_name']}\" {row['data_type']}"
        if row['comment']:
            clean_comment = str(row['comment']).replace("'", "").strip()
            if clean_comment:
                col_def += f" [note: '{clean_comment}']"
        dbml_lines.append(col_def)
    dbml_lines.append("}\n")

# D. Relationships
for _, row in fks_df.iterrows():
    # Clean up formatting for DBML
    src_cols = row['source_cols'].replace("(", "").replace(")", "")
    tgt_cols = row['target_cols'].replace("(", "").replace(")", "")
    
    # If composite (comma exists), keep parens. If single, wrap in quotes.
    src_ref = f"({src_cols})" if "," in src_cols else f"\"{src_cols}\""
    tgt_ref = f"({tgt_cols})" if "," in tgt_cols else f"\"{tgt_cols}\""

    dbml_lines.append(f"Ref {row['constraint_name']}: \"{row['source_table']}\".{src_ref} > \"{row['target_table']}\".{tgt_ref}")

# 4. OUTPUT
final_dbml = "\n".join(dbml_lines)
print(final_dbml)

# 5. WRITE FILE
try:
    with open(output_path, "w") as f:
        f.write(final_dbml)
    print(f"\nSaved to: {output_path}")
except Exception as e:
    print(f"Error writing file: {e}")